<a href="https://colab.research.google.com/github/elijahmflomo/Sem_2_GENERATIVE-AI/blob/main/Assignment_Number_8_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Required Libraries
# Run this cell first (click the play button)

!pip install transformers datasets peft accelerate evaluate scikit-learn
!pip install torch torchvision torchaudio
!pip install gradio  # For easy deployment demo

print(" All libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
 All libraries installed successfully!


**Step 2: Data Preparation **

In [3]:
# @title  2. Load and Prepare Data
from datasets import load_dataset
import pandas as pd
import numpy as np

# Load a smaller toxicity dataset (perfect for beginners)
# This dataset contains text labeled as "toxic" or "not toxic"
print("Loading dataset...")
dataset = load_dataset("tweet_eval", "hate")

# Let's see what's in our data
print("\n Dataset structure:")
print(dataset)

# Convert to pandas to see actual examples
train_df = pd.DataFrame(dataset['train'])
print("\n First 5 examples from training data:")
print(train_df.head())

print("\n Data distribution:")
print(train_df['label'].value_counts())
# 0 = Not toxic, 1 = Toxic

# Let's see some examples
print("\n Sample toxic text (label=1):")
toxic_examples = train_df[train_df['label']==1]['text'].iloc[:3]
for i, text in enumerate(toxic_examples):
    print(f"{i+1}. {text}")

print("\n Good sample non-toxic text (label=0):")
nontoxic_examples = train_df[train_df['label']==0]['text'].iloc[:3]
for i, text in enumerate(nontoxic_examples):
    print(f"{i+1}. {text}")

Loading dataset...


hate/train-00000-of-00001.parquet:   0%|          | 0.00/816k [00:00<?, ?B/s]

hate/test-00000-of-00001.parquet:   0%|          | 0.00/278k [00:00<?, ?B/s]

hate/validation-00000-of-00001.parquet:   0%|          | 0.00/103k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2970 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]


 Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2970
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1000
    })
})

 First 5 examples from training data:
                                                text  label
0  @user nice new signage. Are you not concerned ...      0
1  A woman who you fucked multiple times saying y...      1
2  @user @user real talk do you have eyes or were...      1
3  your girlfriend lookin at me like a groupie in...      1
4                        Hysterical woman like @user      0

 Data distribution:
label
0    5217
1    3783
Name: count, dtype: int64

 Sample toxic text (label=1):
1. A woman who you fucked multiple times saying yo dick small is a compliment you know u hit that spot 😎
2. @user @user real talk do you have eyes or were they gouged out by a rapefugee?
3. y

Step 3: Model Selection (Choosing BERT)

In [4]:
# @title  3. Load Pre-trained Model
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Model selection explanation:
# - BERT is older but simpler to learn with
# - It's small enough to run on free Colab GPU
# - Perfect for classification tasks like content moderation

model_name = "bert-base-uncased"  # Pre-trained BERT model

print(f"Loading tokenizer and model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2  # 2 classes: toxic or not toxic
)

print(" Model loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")

Loading tokenizer and model: bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 Model loaded successfully!
Model parameters: 109,483,778


**Step 5: PEFT Fine-Tuning (Parameter Efficient)**

In [6]:
# @title  5. Setup PEFT Fine-Tuning
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

# Define LoRA configuration
# This adds small trainable adapters to our model
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # Sequence Classification
    r=8,                         # Rank (smaller = more efficient)
    lora_alpha=32,               # Scaling factor
    lora_dropout=0.1,            # Prevent overfitting
    bias="none",
    target_modules=["query", "value"]  # Which parts to adapt
)

# Apply LoRA to our model
peft_model = get_peft_model(model, lora_config)

print("PEFT configuration applied!")
print(f"Trainable parameters: {peft_model.num_parameters(only_trainable=True):,}")
print(f"Total parameters: {peft_model.num_parameters():,}")
print(f"Percentage trainable: {100 * peft_model.num_parameters(only_trainable=True) / peft_model.num_parameters():.2f}%")

# Set up training arguments
training_args = TrainingArguments(
    output_dir="./content-moderation-model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"  # Disable wandb etc.
)

# Define evaluation metrics
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=predictions, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=predictions, references=labels, average="binary")["f1"]
    }

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


PEFT configuration applied!
Trainable parameters: 296,450
Total parameters: 109,780,228
Percentage trainable: 0.27%


**Step 6: Train the Model**

In [9]:
# @title Fine-Tune the Model

# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=128)

# Apply tokenization to the dataset
# The dataset variable is available from cell avaH8_O8Goy0
# The tokenizer variable is available from cell D3RTqiWSHIe-
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Create smaller datasets for faster training (using 20% of data)
small_train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(500))
small_eval_dataset = tokenized_dataset["validation"].shuffle(seed=42).select(range(200))

# Initialize trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

# Start training!
print(" Starting fine-tuning...")
print("This will take 5-10 minutes on Colab GPU")
print("-" * 50)

trainer.train()

print("\n Training complete!")

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2970 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

 Starting fine-tuning...
This will take 5-10 minutes on Colab GPU
--------------------------------------------------


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.708651,0.540000,0.250000,0.022727,0.041667
2,No log,0.704665,0.540000,0.250000,0.022727,0.041667
3,No log,0.703871,0.540000,0.250000,0.022727,0.041667



 Training complete!


**Step 7: Evaluate the Model**

In [11]:
# Evaluate on test set
test_results = trainer.evaluate(tokenized_dataset["test"])
print("\n Model Performance on Test Set:")
print("-" * 30)
for metric, value in test_results.items():
    if "eval_" in metric:
        print(f"{metric.replace('eval_', '')}: {value:.4f}")

# Test with custom examples
def predict_toxicity(texts):
    """Predict if texts are toxic"""
    # Tokenize
    inputs = tokenizer(texts, padding=True, truncation=True,
                       max_length=128, return_tensors="pt")

    # Move inputs to the same device as the model
    inputs = {k: v.to(peft_model.device) for k, v in inputs.items()}

    # Get predictions
    with torch.no_grad():
        outputs = peft_model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

    return predictions.cpu().numpy()

# Test examples
test_texts = [
    "I love this product! Great job everyone!",
    "You are so stupid and worthless, go kill yourself",
    "The weather is nice today, let's go for a walk",
    "I hope you die in a fire you disgusting piece of trash",
]

print("\n Testing with custom examples:")
print("-" * 40)

predictions = predict_toxicity(test_texts)

for text, pred in zip(test_texts, predictions):
    toxicity_score = pred[1]  # Index 1 is toxic class
    label = " TOXIC" if toxicity_score > 0.5 else " SAFE"
    print(f"Text: {text}")
    print(f"Prediction: {label} (confidence: {toxicity_score:.3f})")
    print()


 Model Performance on Test Set:
------------------------------
loss: 0.7021
accuracy: 0.5495
precision: 0.3576
recall: 0.0863
f1: 0.1390
runtime: 23.5783
samples_per_second: 125.9630
steps_per_second: 7.8890

 Testing with custom examples:
----------------------------------------
Text: I love this product! Great job everyone!
Prediction:  SAFE (confidence: 0.394)

Text: You are so stupid and worthless, go kill yourself
Prediction:  SAFE (confidence: 0.422)

Text: The weather is nice today, let's go for a walk
Prediction:  SAFE (confidence: 0.343)

Text: I hope you die in a fire you disgusting piece of trash
Prediction:  SAFE (confidence: 0.430)



**Step 8: Save and Deploy**

In [12]:
import pickle
import json

# Save the model
print("Saving model...")
peft_model.save_pretrained("./final_content_moderation_model")
tokenizer.save_pretrained("./final_content_moderation_model")

# Create a simple prediction function
def moderate_content(text, threshold=0.5):
    """
    Main content moderation function

    Args:
        text: String to moderate
        threshold: Confidence threshold (0-1)

    Returns:
        Dictionary with prediction and confidence
    """
    # Get prediction
    pred = predict_toxicity([text])[0]
    toxicity_score = float(pred[1])

    # Determine result
    is_toxic = toxicity_score > threshold

    return {
        "text": text,
        "is_toxic": is_toxic,
        "toxicity_score": toxicity_score,
        "threshold_used": threshold,
        "action": "BLOCK" if is_toxic else "ALLOW"
    }

# Test the function
test_case = "This is a wonderful community!"
result = moderate_content(test_case)
print("\n API Test Result:")
print(json.dumps(result, indent=2))

Saving model...

 API Test Result:
{
  "text": "This is a wonderful community!",
  "is_toxic": false,
  "toxicity_score": 0.38796550035476685,
  "threshold_used": 0.5,
  "action": "ALLOW"
}


**Step 9: Deploy with Gradio (Simple Web App)**

*   List item
*   List item



In [15]:

import gradio as gr

def moderate_interface(text, threshold):
    """Wrapper for Gradio interface"""
    result = moderate_content(text, threshold)

    if result['is_toxic']:
        return f" BLOCKED\nToxicity Score: {result['toxicity_score']:.3f}"
    else:
        return f" ALLOWED\nToxicity Score: {result['toxicity_score']:.3f}"

# Create the interface
iface = gr.Interface(
    fn=moderate_interface,
    inputs=[
        gr.Textbox(label="Enter text to moderate", lines=3),
        gr.Slider(0.1, 0.9, value=0.5, label="Sensitivity Threshold")
    ],
    outputs=gr.Textbox(label="Moderation Result", lines=2),
    title="AI Content Moderator",
    description="Enter text to check if it contains toxic content",
    examples=[
        ["I love learning about AI!", 0.5],
        ["You are stupid and ugly", 0.5],
        ["Great work team!", 0.7],
    ]
)

# Launch the app
print(" Starting Gradio interface...")
print("Click the link below to open in browser:")
iface.launch(share=True)  # share=True creates a public link

 Starting Gradio interface...
Click the link below to open in browser:
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://114548201f54cb7420.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Step 10: Human-in-the-Loop Pipeline**

In [16]:
import time
from datetime import datetime

class HumanInTheLoopSystem:
    """Simple system for human review of flagged content"""

    def __init__(self):
        self.flagged_content = []
        self.reviewed_content = []

    def process_content(self, text, auto_threshold=0.5, human_threshold=0.3):
        """
        Process content with human review for borderline cases

        Args:
            text: Content to moderate
            auto_threshold: Threshold for auto-blocking
            human_threshold: Lower threshold for human review
        """
        pred = predict_toxicity([text])[0]
        toxicity_score = pred[1]

        content_id = len(self.flagged_content) + len(self.reviewed_content)

        if toxicity_score > auto_threshold:
            # Automatically block
            decision = {
                "id": content_id,
                "text": text,
                "score": float(toxicity_score),
                "decision": "AUTO_BLOCK",
                "timestamp": datetime.now().isoformat()
            }
            self.reviewed_content.append(decision)
            return decision

        elif toxicity_score > human_threshold:
            # Send for human review
            flagged = {
                "id": content_id,
                "text": text,
                "score": float(toxicity_score),
                "status": "PENDING_REVIEW",
                "timestamp": datetime.now().isoformat()
            }
            self.flagged_content.append(flagged)
            return flagged
        else:
            # Automatically allow
            decision = {
                "id": content_id,
                "text": text,
                "score": float(toxicity_score),
                "decision": "AUTO_ALLOW",
                "timestamp": datetime.now().isoformat()
            }
            self.reviewed_content.append(decision)
            return decision

    def review_content(self, content_id, human_decision):
        """Human reviews flagged content"""
        for item in self.flagged_content:
            if item["id"] == content_id:
                item["status"] = "REVIEWED"
                item["human_decision"] = human_decision
                self.reviewed_content.append(item.copy())
                self.flagged_content.remove(item)
                return True
        return False

    def get_stats(self):
        """Get system statistics"""
        return {
            "total_processed": len(self.reviewed_content),
            "pending_review": len(self.flagged_content),
            "auto_blocked": sum(1 for x in self.reviewed_content if x.get("decision") == "AUTO_BLOCK"),
            "auto_allowed": sum(1 for x in self.reviewed_content if x.get("decision") == "AUTO_ALLOW"),
            "human_reviewed": sum(1 for x in self.reviewed_content if "human_decision" in x)
        }

# Demo the human-in-the-loop system
hitl = HumanInTheLoopSystem()

print(" Testing Human-in-the-Loop System")
print("-" * 40)

test_contents = [
    "I love this community!",
    "You are an idiot",
    "This is a slightly problematic comment",
    "Great work everyone!"
]

for text in test_contents:
    result = hitl.process_content(text, auto_threshold=0.7, human_threshold=0.3)
    print(f"\nText: {text}")
    print(f"Result: {result}")

print("\n System Statistics:")
print(hitl.get_stats())

 Testing Human-in-the-Loop System
----------------------------------------

Text: I love this community!
Result: {'id': 0, 'text': 'I love this community!', 'score': 0.36240559816360474, 'status': 'PENDING_REVIEW', 'timestamp': '2026-03-14T16:36:51.674836'}

Text: You are an idiot
Result: {'id': 1, 'text': 'You are an idiot', 'score': 0.5148576498031616, 'status': 'PENDING_REVIEW', 'timestamp': '2026-03-14T16:36:51.711770'}

Text: This is a slightly problematic comment
Result: {'id': 2, 'text': 'This is a slightly problematic comment', 'score': 0.544962465763092, 'status': 'PENDING_REVIEW', 'timestamp': '2026-03-14T16:36:51.739504'}

Text: Great work everyone!
Result: {'id': 3, 'text': 'Great work everyone!', 'score': 0.4120091199874878, 'status': 'PENDING_REVIEW', 'timestamp': '2026-03-14T16:36:51.761820'}

 System Statistics:
{'total_processed': 0, 'pending_review': 4, 'auto_blocked': 0, 'auto_allowed': 0, 'human_reviewed': 0}
